## Final Model

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import joblib
import os

from src.data_loader import load_raw_data
from src.preprocessing import run_preprocessing_pipeline

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, f1_score

df = load_raw_data()
splits, artifacts = run_preprocessing_pipeline(df)

class_weights = compute_class_weight(
    'balanced', classes=np.unique(splits.y_train), y=splits.y_train
)
class_weight_dict = dict(enumerate(class_weights))


def build_and_train(hidden_units, dropout_rate, learning_rate, verbose=0):
    model = keras.Sequential([layers.Input(shape=(splits.X_train.shape[1],))])
    for units in hidden_units:
        model.add(layers.Dense(units, activation='relu'))
        model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )
    model.fit(
        splits.X_train, splits.y_train,
        validation_data=(splits.X_val, splits.y_val),
        epochs=50, batch_size=256, class_weight=class_weight_dict,
        callbacks=[early_stop], verbose=verbose
    )
    y_proba = model.predict(splits.X_val, verbose=0).ravel()
    y_pred = (y_proba >= 0.5).astype(int)
    return model, roc_auc_score(splits.y_val, y_proba), f1_score(splits.y_val, y_pred)

Loaded data via local (30000 rows, 25 columns).


In [2]:
import os
import joblib

# Retrain the winning configuration from the tuning grid: [64,32], dropout=0.3
final_model, final_auc, final_f1 = build_and_train(
    hidden_units=[64, 32], dropout_rate=0.3, learning_rate=0.001
)
print(f"Final model — ROC-AUC: {final_auc:.4f}, F1: {final_f1:.4f}")

os.makedirs("../models", exist_ok=True)

# Save the model itself
final_model.save("../models/final_nn_model.keras")

# Save the scaler — required to transform any new/raw input the same
# way training data was transformed, before it reaches the model
joblib.dump(artifacts.scaler, "../models/scaler.pkl")

# Save the log-transform shifts — required so new input gets the same
# skew-correction transform applied to training data (see preprocessing.py)
joblib.dump(artifacts.log_shifts, "../models/log_shifts.pkl")

# Save the exact feature column order the model expects — mismatched
# order is a silent-failure bug: no error, just wrong predictions
joblib.dump(artifacts.feature_columns, "../models/feature_columns.pkl")

print("Saved: final_nn_model.keras, scaler.pkl, log_shifts.pkl, feature_columns.pkl")

Final model — ROC-AUC: 0.7761, F1: 0.5349
Saved: final_nn_model.keras, scaler.pkl, log_shifts.pkl, feature_columns.pkl


In [3]:
from tensorflow import keras as k

reloaded_model = k.models.load_model("../models/final_nn_model.keras")
reloaded_scaler = joblib.load("../models/scaler.pkl")
reloaded_cols = joblib.load("../models/feature_columns.pkl")

sample_pred = reloaded_model.predict(splits.X_val.iloc[:3], verbose=0)
print("Reloaded model sample predictions:", sample_pred.ravel())
print("Feature columns match:", reloaded_cols == list(splits.X_train.columns))

Reloaded model sample predictions: [0.6977454  0.73149556 0.5933225 ]
Feature columns match: True
